<a href="https://colab.research.google.com/github/raphhax/deep-learning/blob/main/primeirospassosdl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

O conteúdo das seguintes células são baseadas nos conhecimentos adquiridos
durante a aula 1 do curso de PyTorch e Deep Learning do canal Programação Dinâmica e
implementam redes neurais usando regressão linear

https://www.youtube.com/watch?v=cGxv8tOaA7I&t=3800s$0


In [1]:
import torch
import numpy as np
from torch import nn

In [2]:
class LineNetwork(nn.Module):
# Inicializando
  def __init__(self):
    # Construtor da classe de redes neurais
    super().__init__()
    #self.layers = nn.Linear(1,1) # Usamos essa quando vamos usar apenas 1 camada, mas o layers que vamos usar nesse projeto tem várias camadas, por isso método sequential.
    self.layers = nn.Sequential(
        nn.Linear(1,1)
    )

# Como a rede computa
  def forward(self, x):
    return self.layers(x)


In [3]:
from torch.utils.data import Dataset, DataLoader
import torch.distributions.uniform as urand

In [4]:
# Cria base de dados que vamos usar
class AlgebraicDataset(Dataset): # Nosso dataset deriva do dataset do pytorch ent ajuda no dataloader
  def __init__(self, funcao, intervalo, nsamples):  #esse eh os parametros do meu dataset
    X = urand.Uniform(intervalo[0], intervalo[1].sample([nsamples])) # gero um x com varios valores aleatorios
    self.data = [(x,funcao(x)) for x in X] # meus dados viram um par ordenado dos valores aleatorios gerados

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    return self.data[index]

In [5]:
line = lambda x: 2*x + 3 # funcao que usamos no dataset algebrico
intervalo = (-10, 10)
train_nsamples = 1000 # quantidade de amostras para usar na previsao

test_nsamples = 100

In [6]:
# Carrega a nossa base de dados
train_dataset = AlgebraicDataset(line, intervalo, train_nsamples)
test_dataset = AlgebraicDataset(line, intervalo, test_nsamples)

train_dataloader = DataLoader(train_dataset, batch_size=train_nsamples, shuffle=True)
#batch_size : serve para ler de pouquinho em poquinho principalmente para grandes volumes, mas como o nosso eh pouco, vamos ler tudo de cada vez (=train_nsamples)
#shuffle=True: serve para TRUE embaralhar cada conjuntinho de dados do zero para os minipacotes lidos de acordo com o batch_size, ajuda na variabilidade dos dados

test_dataloader = DataLoader(test_dataset, batch_size=test_nsamples, shuffle=True)



AttributeError: 'int' object has no attribute 'sample'

Hiperparâmetros de Otimização

In [ ]:
# serve para saber onde ta rodando o código e o pytorch consegue identificar onde
# cuda eh gpu, e gpu é muito melhor e mais rápido que cpu, tipo para imagens
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Rodando na {device}")

In [ ]:
# Cria meu modelo passando gpu ou cpu, nesse caso cpu
model = LineNetwork().to(device)

In [ ]:
#aqui vamos começar a corrigir o erro
# Função de perda
# Erro Quadrático Médio eh oq usaremos
lossfunc = nn.MSELoss()

# Gradiente Descendente Estocástico
# SGD = Stochastic Gradient Descent
# Método guloso que não percebe se esta no mínimo global ou local
# Calculamos o erro para uma aprte do conjunto de dados, que eh aquele pedaço que passamos no batch_size no DataLoader
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) # lr= taxa de aprendizado, no caso 0,001

In [ ]:
def train(model, dataloader, lossfunc, optimizer):
  model.train()
  cumloss = 0.0
  for X, y in dataloader: # X é o dado e Y é o rótulo/resposta/verdade que quero aprender a pré-dizer
    X = X.unsqueeze(1).float().to(device) # faz o vetor virar tensores de 1 dimensão, com cada valor desse vetor x virando um vetor separado dentro desse vetor principal
    y = y.unsqueeze(1).float().to(device)

    pred = model(X)
    loss = lossfunc(pred, y) # passa o que q eu prediz com o 'pred' e oq que eh o valor real com 'y'

    # zera os gradientes acumulados, mas no geral vc nao quer acumular mesmo
    optimizer.zero_grad()
    # computa os gradientes
    loss.backward() # retroprograpagação com os gradientes de uma camada para a outra, usa regra da cadeia pra ir de trás pra frente
    # anda, de fato, na direcao que reduz o erro local
    optimizer.step()

    cumloss += loss.item() # usa o item pra pegar o float do gradiente pq loss eh um tensor de 1 dimensao

  return cumloss/len(dataloader)

def test(model, dataloader, lossfunc):
  model.eval() # forma de só avaliar o modelo
  cumloss = 0.0
  with torch.no_grad(): # para falar q nao queremos acumular gradientes
    for X, y in dataloader: # X é o dado e Y é o rótulo/resposta/verdade que quero aprender a pré-dizer
      X = X.unsqueeze(1).float().to(device) # faz o vetor virar tensores de 1 dimensão, com cada valor desse vetor x virando um vetor separado dentro desse vetor principal
      y = y.unsqueeze(1).float().to(device)

      pred = model(X)
      loss = lossfunc(pred, y) # passa o que q eu prediz com o 'pred' e oq que eh o valor real com 'y'
      cumloss += loss.item() # usa o item pra pegar o float do gradiente pq loss eh um tensor de 1 dimensao


  return cumloss/len(dataloader)

Treinando a rede

In [ ]:
epochs = 101 # quantidade de vezes que vamos treinar a rede
for t in range(epochs):
  train_loss = train(model, train_dataloader, lossfunc, optimizer) #erro de treinamento
  if t % 10 == 0:
    print(f"Epoch: {t}; Train Loss: {train_loss}")

test_loss = test(model, test_dataloader, lossfunc)
print(f"Test Loss: {test_loss}")